# xarray with terrio

`terrio.xarray` brings [rioxarray](https://corteva.github.io/rioxarray/)'s API to [terrio](https://terrio.dev), without installing GDAL. This notebook runs entirely in your browser, in [JupyterLite](https://jupyterlite.readthedocs.io).

Browser builds of terrio do not open URLs, so this notebook downloads its data first. To open cloud-optimized GeoTIFFs lazily over HTTP, run the [full notebook](https://github.com/opengeos/terrio/blob/main/docs/notebooks/xarray.ipynb) in Google Colab or locally.

## Install

In [ ]:
%pip install -q terrio xarray dask matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import terrio
import terrio.xarray as txr

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["figure.dpi"] = 72
terrio.__version__

## Download the data

In [ ]:
from pyodide.http import pyfetch


async def download(url, path):
    """Saves a URL to a file in the browser's memory."""
    response = await pyfetch(url)
    with open(path, "wb") as file:
        file.write(await response.bytes())


data = "https://raw.githubusercontent.com/opengeos/data/main"

In [ ]:
await download(f"{data}/raster/srtm90.tif", "srtm90.tif")
await download(f"{data}/raster/landsat7.tif", "landsat7.tif")

## A raster as a DataArray

An SRTM elevation model of the Sierra Nevada. `masked=True` turns any nodata into NaN; `chunks=True` makes a dask array with chunks aligned on the file's tiles.

In [ ]:
dem = txr.open_rasterio("srtm90.tif", masked=True, chunks=True)
dem

Coordinates are pixel centres; the `spatial_ref` coordinate holds the CRS and the geotransform, as rioxarray writes them, and the data type is in the encoding (with the nodata value of a file that declares one):

In [ ]:
print(dem.spatial_ref.attrs["grid_mapping_name"], dem.spatial_ref.attrs["GeoTransform"])
print(dem.encoding["dtype"], dem.chunks)
print(dem.terrio.crs, dem.terrio.resolution(), dem.terrio.bounds())

## A window, by coordinates

`clip_box` selects a window by coordinates.

In [ ]:
yosemite = dem.terrio.clip_box(-119.75, 37.6, -119.3, 37.95, crs="EPSG:4326")
print(yosemite.shape)
yosemite.squeeze().plot.imshow(cmap="terrain")
plt.title("Yosemite National Park");

## An overview

`overview_level` opens one of the file's overviews: a quick look at the whole image.

In [ ]:
overview = txr.open_rasterio("srtm90.tif", overview_level=2, masked=True)
print(overview.shape, overview.terrio.resolution())
overview.squeeze().plot.imshow(cmap="terrain");

## Reprojecting

`reproject` warps with terrio's implementation of GDAL's warper. `estimate_utm_crs` picks the UTM zone of the data.

In [ ]:
utm_crs = yosemite.terrio.estimate_utm_crs()
utm = yosemite.terrio.reproject(utm_crs, resolution=90, resampling="bilinear")
print(utm_crs, utm.shape, utm.terrio.resolution())
utm.squeeze().plot.imshow(cmap="terrain");

## Clipping to a polygon

Geometries are GeoJSON-like dicts (or shapely geometries), in the raster's CRS or in `crs`. Pixels outside become NaN (or nodata).

In [ ]:
import math

lon0, lat0, r = -119.525, 37.775, 0.12
ring = [(lon0 + r * math.cos(t / 20 * 2 * math.pi), lat0 + r * math.sin(t / 20 * 2 * math.pi)) for t in range(21)]
circle = {"type": "Polygon", "coordinates": [ring]}
clipped = yosemite.terrio.clip([circle], crs="EPSG:4326")
clipped.squeeze().plot.imshow(cmap="terrain");

## The xarray backend

`xr.open_dataset(..., engine="terrio")` opens rasters as Datasets, with `mask_and_scale=True` by default as rioxarray's `rasterio` engine (nodata pixels become NaN). A three-band Landsat 7 image, kept as bytes here:

In [ ]:
landsat = xr.open_dataset("landsat7.tif", engine="terrio", mask_and_scale=False)
landsat

In [ ]:
landsat.band_data.plot.imshow(rgb="band");

## Writing

`to_raster` writes GeoTIFFs (`driver="COG"` for cloud-optimized ones) and the other formats terrio writes, with the CRS, transform, nodata, scales, offsets, band descriptions and attributes.

In [ ]:
# Reprojecting left NaN in the masked data, so write floats.
utm.terrio.to_raster("yosemite_utm.tif", driver="COG", compress="DEFLATE", dtype="float32")
back = txr.open_rasterio("yosemite_utm.tif", masked=True)
print(back.terrio.crs, back.shape, bool(np.allclose(back.values, utm.values, equal_nan=True)))

## Code written for rioxarray

`register_rio_accessor()` registers the accessor as `.rio` too (when rioxarray is not imported), so that rioxarray code runs unchanged:

In [ ]:
txr.register_rio_accessor()
sub = dem.rio.clip_box(-119.6, 37.7, -119.5, 37.8, crs="EPSG:4326")
print(sub.rio.crs, sub.rio.shape, float(sub.mean()))